# Sporet ESRI data
Sporet is Skiforeninges new Web/App for cross country skiing trail status. It is a merge of old Skiforeningen web/app Imarka and Skisporet.no.


Sporet is implemented by Geodata in ESRI ArcGIS

<https://developers.arcgis.com/rest/services-reference/enterprise/catalog/>

### Proxy Sporet
<https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer>

* 0 Ski trail segment parts <https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/0/query?f=json>
* 1 Ski trail segments <https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/1/query?f=json>
* 2 Vehicle positions <https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/2/query?f=json>
* 3 Ski trail segment parts zoom <https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/3/query?f=json>

### Markadatabase_v2 
<https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2>

* 2 POIer - <https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/2/query?f=json>
* 4 Destinasjoner_prep - <https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/4/query?f=json>
* 5 Sykkelveier (5)
* 6 Loypetyper (6)
* 8 Holdeplasser - <https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/8/query?f=json>
* 11 Destinasjoner_singel - <https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/11/query?f=json>
* 13 Infopoint - <https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/13/query?f=json>


In [ ]:
import requests

import geopandas as gpd
import pandas as pd
from lonboard import viz
import duckdb

from shapely import LineString, MultiLineString

## Proxy
### Vehicle positions (layer 2)

```https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/2/query?f=json```

This last part got cut off
```
&geometry=%7B%22xmin%22%3A225030.61127150018%2C%22ymin%22%3A6643295.002319505%2C%22xmax%22%3A229922.5810817502%2C%22ymax%22%3A6648186.972129756%7D&
orderByFields=vehicleid+ASC&
outFields=*
&outSR=25833&
resultType=tile
&returnExceededLimitFeatures=false
&spatialRel=esriSpatialRelIntersects
&where=is_visible+%3D+1&
geometryType=esriGeometryEnvelope&
inSR=25833&
returnAdvancedSymbols=true
```

```json
        {
            "attributes": {
                "course": 358,
                "is_visible": 0,
                "lastseen": 1771772667000,
                "name": "Alta kommune PB100-1",
                "orgid": 10278,
                "renderer_value": "stopped_groomer",
                "speed": 0,
                "vehicleid": 10232
            },
            "geometry": {
                "x": 819328.38,
                "y": 7782854.06
            }
        },

```


In [ ]:
res = requests.get("https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/2/query?f=json&outSr=4326")
vehicles_json = res.json()
print(vehicles_json["features"][0:2])
print(len(vehicles_json["features"]))

In [ ]:
print(vehicles_json)

In [ ]:
df_vehicles = (
    pd.json_normalize(data=vehicles_json["features"], sep="_")
    .rename(columns=lambda c: c.replace("attributes_", "").replace("geometry_", ""))
    .rename(columns={"x": "lon", "y": "lat", "objectid": "id"})
    .assign(lastseen=lambda df: pd.to_datetime(df["lastseen"], unit="ms"))
)
df_vehicles

In [ ]:
gdf_vehicles = gpd.GeoDataFrame(df_vehicles, geometry=gpd.points_from_xy(df_vehicles.lon, df_vehicles.lat), crs="EPSG:4326")
gdf_vehicles.explore()

https "https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/2/query?f=json&where=1=1&outFields=*&outSR=4326"

```https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/0/query?f=json&geometry=%7B%22xmin%22%3A225030.61127150018%2C%22ymin%22%3A6643295.002319505%2C%22xmax%22%3A229922.5810817502%2C%22ymax%22%3A6648186.972129756%7D&maxAllowableOffset=9.55462853564454&orderByFields=id%20ASC&outFields=has_skating%2Cid%2Cparentsegmentid%2Cprepsymbol&outSR=25833&resultType=tile&returnExceededLimitFeatures=false&spatialRel=esriSpatialRelIntersects&where=1%3D1&geometryType=esriGeometryEnvelope&inSR=25833&returnAdvancedSymbols=true```

And similar for this
```https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/0/query?f=json&geometry=%7B%22xmin%22%3A229922.5810817502%2C%22ymin%22%3A6643295.002319505%2C%22xmax%22%3A234814.5508920002%2C%22ymax%22%3A6648186.972129756%7D&maxAllowableOffset=9.55462853564454&orderByFields=id%20ASC&outFields=has_skating%2Cid%2Cparentsegmentid%2Cprepsymbol&outSR=25833&resultType=tile&returnExceededLimitFeatures=false&spatialRel=esriSpatialRelIntersects&where=1%3D1&geometryType=esriGeometryEnvelope&inSR=25833&returnAdvancedSymbols=true```


Got 49 of these
```
{
	"0": {
		"attributes": {
			"id": 2831191,
			"prepsymbol": 50,
			"parentsegmentid": 134717,
			"has_skating": 0
		},
		"geometry": {
			"paths": [
				[
					[
						225906.62,
						6644182.14
					],
					[
						225974.83,
						6644197.5
					],
					[
						226059.85,
						6644301.01
					]
				]
			]
		}
	},
    {
	"1": {
		"attributes": {
			"id": 2831192,
			"prepsymbol": 50,
			"parentsegmentid": 134717,
			"has_skating": 0
		},
		"geometry": {
			"paths": [
				[
					[
						226059.85,
						6644301.01
					],
					[
						226061.37,
						6644302.87
					],
					[
						226093.62,
						6644434.61
					],
					[
						226094.34,
						6644500.45
					]
				]
			]
		}
	}
}
}

In [ ]:
res = requests.get((
    "https://maps.sporet.no/arcgis/rest/services/Proxy/Sporet/FeatureServer/0/query?"
    "f=json&"
    # "inSR=25833&"
    # 'geometry={"xmin":225030.61,"ymin":6643295.00,"xmax":229922.58,"ymax":6648186.97}&'
    "inSR=4326&"
    'geometry={"xmin":10.30,"ymin":59.58,"xmax":10.65,"ymax":59.98}&'
    "maxAllowableOffset=9.55462853564454&"
    "orderByFields=id%20ASC&"
    "outFields=has_skating%2Cid%2Cparentsegmentid%2Cprepsymbol&"
    "outSr=4326&"
    "resultType=tile&"
    "returnExceededLimitFeatures=false&"
    "spatialRel=esriSpatialRelIntersects&"
    "where=1%3D1&"
    "geometryType=esriGeometryEnvelope&"
    "returnAdvancedSymbols=true")
    )
tracks_json = res.json()
print(tracks_json["features"][0:2])
print(len(tracks_json["features"]))

In [ ]:
gdf_tracks = (
    pd.json_normalize(tracks_json["features"], sep='_')
    .rename(columns=lambda c: c.replace('attributes_', '').replace('geometry_', ''))
    .assign(geometry=lambda df: df['paths'].map(
        lambda paths: MultiLineString([LineString(p) for p in paths])
    ))
    .drop(columns=['paths'])
    .pipe(gpd.GeoDataFrame, crs='EPSG:4326')
    .to_crs('EPSG:4326')  # reproject to WGS84 if needed
)
gdf_tracks.explore()


## Protobuf
### Warnings
https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/13/query?f=pbf&cacheHint=true&maxRecordCountFactor=4&resultOffset=0&resultRecordCount=5000&where=1%3D1&orderByFields=segmentid%2Cwarningtext%20ASC&outFields=*&outSR=25833&spatialRel=esriSpatialRelIntersects&returnAdvancedSymbols=true


Just change format to json

```json
        {
            "attributes": {
                "ESRI_OID": 87,
                "segmentid": 145754,
                "warningtext": "For lite snø og mye overvann. kun kjørt forberedende arbeid. Preppes ikke med spor enda."
            },
            "geometry": {
                "x": 515422.4321999997,
                "y": 8686541.1501
            }
        },

```


In [ ]:
res = requests.get("https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/13/query?f=json&cacheHint=true&maxRecordCountFactor=4&resultOffset=0&resultRecordCount=5000&where=1%3D1&orderByFields=segmentid%2Cwarningtext%20ASC&outFields=*&outSR=4326&spatialRel=esriSpatialRelIntersects&returnAdvancedSymbols=true")
warnings_json = res.json()

warnings_json["features"][0:2]

In [ ]:
gdf_warnings = (
    pd.json_normalize(data=warnings_json["features"], sep="_")
    .rename(columns=lambda c: c.replace("attributes_", "").replace("geometry_", ""))
    .assign(geometry=lambda df_: gpd.points_from_xy(df_["x"], df_["y"]))
    .drop(columns=["x", "y"])
    .rename(columns={"ESRI_OID": "id"})
    .pipe(gpd.GeoDataFrame, crs='EPSG:4326')
)
gdf_warnings.explore()

## Hytter
```https "https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/2/query?f=json&cacheHint=true&maxRecordCountFactor=4&resultOffset=0&resultRecordCount=5000&where=poitypeid%20IN%20(%27ACF%27%2C%27CAF%27%2C%27CAM%27)&orderByFields=id%20ASC&outFields=*&outSR=25833&spatialRel=esriSpatialRelIntersects&returnAdvancedSymbols=true"```

```json
        {
            "attributes": {
                "destinationid": 10003,
                "id": 16940,
                "name": "Restaurant Fjellroa",
                "poitypeid": "CAF"
            },
            "geometry": {
                "x": 348722.1486999998,
                "y": 6801492.674400002
            }
        },
        {
            "attributes": {
                "destinationid": 10003,
                "id": 16941,
                "name": "Restaurant Pilegrimen",
                "poitypeid": "CAF"
            },
            "geometry": {
                "x": 347763.4855000004,
                "y": 6802026.517900001
            }
        },

```

### Områder
```https://maps.sporet.no/arcgis/rest/services/Markadatabase_v2/Sporet_Simple/MapServer/11/query?f=json&cacheHint=true&maxRecordCountFactor=4&resultOffset=0&resultRecordCount=5000&where=1%3D1&orderByFields=id%20ASC&outFields=*&outSR=25833&spatialRel=esriSpatialRelIntersects&returnAdvancedSymbols=true```


```json
            "attributes": {
                "id": 13163,
                "name": "Galterud"
            },
            "geometry": {
                "x": 325745.51939999964,
                "y": 6675840.518200001
            }
        },
        {
            "attributes": {
                "id": 13196,
                "name": "Åmot i Modum kommune"
            },
            "geometry": {
                "x": 215821.95610000007,
                "y": 6650288.331700001
            }
        },

```

## Track requests and what differs

```
f=json
geometry=%7B%22xmin%22%3A225030.61127150018%2C%22ymin%22%3A6643295.002319505%2C%22xmax%22%3A229922.5810817502%2C%22ymax%22%3A6648186.972129756%7D
maxAllowableOffset=9.55462853564454
orderByFields=id%20ASC
outFields=has_skating%2Cid%2Cparentsegmentid%2Cprepsymbol
outSR=25833
resultType=tile
returnExceededLimitFeatures=false
spatialRel=esriSpatialRelIntersects
where=1%3D1
geometryType=esriGeometryEnvelope
inSR=25833
returnAdvancedSymbols=true
```

```
f=json
geometry=%7B%22xmin%22%3A229922.5810817502%2C%22ymin%22%3A6643295.002319505%2C%22xmax%22%3A234814.5508920002%2C%22ymax%22%3A6648186.972129756%7D
maxAllowableOffset=9.55462853564454
orderByFields=id%20ASC
outFields=has_skating%2Cid%2Cparentsegmentid%2Cprepsymbol
outSR=25833
resultType=tile
returnExceededLimitFeatures=false
spatialRel=esriSpatialRelIntersects
where=1%3D1
geometryType=esriGeometryEnvelope
inSR=25833
returnAdvancedSymbols=true
```